In [18]:
import torch
import torch.nn as nn
import sys
sys.dont_write_bytecode = True

In [19]:
import json
with open("../check_points/tiny_stories_vocab.json") as f:
    word2idx = json.load(f)
    print(type(word2idx))
    print(list(word2idx.items())[:10])

<class 'dict'>
[('<pad>', 0), ('<unk>', 1), ('<sos>', 2), ('<eos>', 3), ('"', 4), ('""I', 5), ('""No', 6), ('"\'Hello,', 7), ('"\'Let\'s', 8), ('"\'Why', 9)]


In [20]:
sys.path.insert(0, '../')
from utils import Tokenizer
tokenizer = Tokenizer()
tokenizer.upload_vocab(word2idx)
tokenizer.encode("little")

[33448]

In [21]:
# embedding_dim == hidden_size == (D)
# embedding_dim % num_heads == 0
embedding_dim = 64

ff_embedding_dim = 128 # ff_embedding_dim = 4 × embedding_dim
max_seq_len = 10
dropout = 0.1
num_heads = 2
vocab_size = tokenizer.get_vocab_size()
num_layers = 2
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [22]:
sys.path.insert(0, '../model')
from gpt2 import GPT2Model

model = GPT2Model(vocab_size,embedding_dim,ff_embedding_dim,max_seq_len,num_heads,num_layers,dropout = 0.1)

# model_path = "/kaggle/input/gpt2minimodel/pytorch/default/1/gpt2MiniModel.pt"
model_path = "../tests/model_check_points/checkpoint_epoch_145.pt"

check_point = torch.load(model_path, map_location=device)

model.load_state_dict(check_point['model_state_dict'])

model.to(device)
model.eval()

C:\Users\diaal\AppData\Local\Temp\ipykernel_8676\732782711.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  check_point = torch.load(model_path, map_location=device)


GPT2Model(
  (embeddings): EmbeddingLayer(
    (token_embedding): Embedding(57374, 64)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layers): ModuleList(
    (0-1): 2 x DecoderLayer(
      (attn): ResidualBlock(
        (sub_layer): MultiHeadSelfAttention(
          (qkv_proj): Linear(in_features=64, out_features=192, bias=True)
          (out_proj): Linear(in_features=64, out_features=64, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      )
      (ffn): ResidualBlock(
        (sub_layer): FeedForward(
          (ff): Sequential(
            (0): Linear(in_features=64, out_features=128, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=128, out_features=64, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (ln_f): LayerNorm((64,),

In [23]:
def top_k_logits(logits, k):
    v, ix = torch.topk(logits, k)
    out = logits.clone()
    out[out < v[:, [-1]]] = -float('Inf')
    return out

In [24]:
def generate_text(prompt, max_new_tokens=50, temp=1.0, top_k_val=50):
    input_ids = tokenizer.encode(prompt)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)  # [1, T]
    
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Truncate to last max_seq_len tokens for positional embedding
            if input_tensor.size(1) > max_seq_len:
                input_tensor_trunc = input_tensor[:, -max_seq_len:]
            else:
                input_tensor_trunc = input_tensor
                
            # print(input_tensor_trunc)
            
            logits = model(input_tensor_trunc)  # [1, T, vocab]
            next_token_logits = logits[:, -1, :]  # last position
            next_token_logits = next_token_logits/temp
            next_token_logits = top_k_logits(next_token_logits, top_k_val)

            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1) 

            if next_token.dim() == 1:
                next_token = next_token.unsqueeze(0)
            # next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)  # [1, 1]
            input_tensor = torch.cat([input_tensor, next_token], dim=1)  # grow the sequence

    return tokenizer.decode(input_tensor[0].tolist())

In [25]:
print(tokenizer.encode("one day a cat went to school"))

[37567, 21502, 12199, 17846, 54947, 51773, 44465]


In [35]:
prompt = "once upon a time in a land far, far away there was a little cat"
print(generate_text(prompt, max_new_tokens=10))

once upon a time in a land far, far away there was a little cat girl named Lucy. was named Lily She had had a
